# 05 MLflow Tracking and Registry

This notebook reviews training runs, registers existing local adapters, and compares adapter experiment metadata.

```mermaid
flowchart LR
    A[Training notebooks] --> B[MLflow experiment]
    B --> C[Run params and metrics]
    B --> D[Adapter artifacts]
    D --> E[MLflow Model Registry]
```

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Configure MLflow

For local notebooks, file-backed MLflow works without a service. With `make up`, set `MLFLOW_TRACKING_URI=http://localhost:5000`.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings_cfg.mlflow_tracking_uri)
mlflow.set_experiment(settings_cfg.mlflow_experiment_name)
client = MlflowClient()
experiment = client.get_experiment_by_name(settings_cfg.mlflow_experiment_name)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else "not created yet")

## Register local adapters

Expected output: one registered model per adapter.

In [ ]:
from training.register_mlflow import register_local_adapter

for adapter in settings_cfg.adapters:

    adapter_path = Path(cfg["output_dir"]) / adapter

    if not adapter_path.exists():
        print(f"Skipping {adapter}: {adapter_path} does not exist yet")
        continue

    # Register adapter
    register_local_adapter(
        adapter,
        adapter_path,
        settings_cfg
    )

    # Full MLflow model name
    model_name = f"qwen2_5_7b_lora_{adapter}"

    # Fetch versions
    latest_versions = client.search_model_versions(
        f"name='{model_name}'"
    )

    latest_version = max(
        int(v.version)
        for v in latest_versions
    )

    print(
        f"Setting prod alias for "
        f"{model_name} v{latest_version}"
    )

    client.set_registered_model_alias(
        name=model_name,
        alias="prod",
        version=latest_version
    )

    print(
        f"✅ {model_name} "
        f"-> prod -> v{latest_version}"
    )

## Compare experiment runs

The table shows recent runs, adapter tags, and common LoRA parameters.

In [ ]:
import pandas as pd

experiment = client.get_experiment_by_name(settings_cfg.mlflow_experiment_name)
if experiment:
    runs = mlflow.search_runs([experiment.experiment_id])
    cols = [
        "run_id",
        "status",
        "tags.adapter_name",
        "params.lora_r",
        "params.learning_rate",
        "metrics.mean_keyword_score",
    ]
    display(runs[[c for c in cols if c in runs.columns]].head(20))
else:
    print("No experiment found yet.")